In [4]:
# numpy and pandas for data manipulation
import numpy as np
import pandas as pd
from sklearn.calibration import LabelEncoder 
import shap
# File system manangement
import os

# Suppress warnings 
import warnings
warnings.filterwarnings('ignore')

# matplotlib and seaborn for plotting
import matplotlib.pyplot as plt
import seaborn as sns

from lightgbm import LGBMClassifier
from yellowbrick.classifier import DiscriminationThreshold

# from sklearnex import patch_sklearn
# patch_sklearn()  # patches scikit-learn algorithms
# from sklearnex import unpatch_sklearn
# unpatch_sklearn()


# ------------------- IMPORT SRC ------------------------------------
# src is the parent folder of notebooks, so we need to add it to sys.path to import config and utils
import sys
notebook_dir = os.getcwd() 

# Parent folder of src
project_root = os.path.abspath(os.path.join(notebook_dir, "..")) 
sys.path.append(project_root)

print("sys.path contains:", sys.path[-1])

from src import config, utils  
# -------------------------------------------------------



# -------------------------------
# Load Data
# -------------------------------
X_train = pd.read_csv('../data/X_train_encoded.csv')
X_test  = pd.read_csv('../data/X_test_encoded.csv')
y_train = pd.read_csv('../data/y_train.csv')
# test_ids = X_test['PassengerId']
# X_test.drop(columns=['PassengerId'], inplace=True)

# Flatten target if needed
# Map target to numeric
target = config.Target

y_train_numeric = y_train[target]


num_classes = len(np.unique(y_train_numeric))
print(f"Number of classes: {num_classes}")

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)






sys.path contains: /home/ismail/Documents/projects/ml-projects/x42
Number of classes: 2
X_train shape: (136429, 13)
X_test shape: (90954, 14)


In [5]:
import torch
import torch.nn as nn

class TorchNet(nn.Module):

    def __init__(self, input_dim):
        super().__init__()

        self.net = nn.Sequential(

            nn.BatchNorm1d(input_dim),

            nn.Linear(input_dim, 32),
            nn.BatchNorm1d(32),
            nn.LeakyReLU(),

            nn.Linear(32, 64),
            nn.BatchNorm1d(64),
            nn.LeakyReLU(),

            nn.Linear(64, 16),
            nn.BatchNorm1d(16),
            nn.LeakyReLU(),

            nn.Linear(16, 4),
            nn.BatchNorm1d(4),
            nn.LeakyReLU(),

            nn.Linear(4, 1)
        )

    def forward(self, x):
        return self.net(x)
    

import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import numpy as np
import torch.optim as optim

def objective(trial):

    lr = trial.suggest_float("lr", 1e-4, 5e-3, log=True)
    batch_size = trial.suggest_categorical("batch_size", [128,256,512])
    epochs = trial.suggest_int("epochs", 5, 25)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    auc_scores = []

    for train_idx, val_idx in skf.split(X_train, y_train_numeric):

        X_tr = torch.tensor(X_train.iloc[train_idx].values, dtype=torch.float32)
        y_tr = torch.tensor(y_train_numeric.iloc[train_idx].values, dtype=torch.float32).view(-1,1)

        X_val = torch.tensor(X_train.iloc[val_idx].values, dtype=torch.float32)
        y_val = torch.tensor(y_train_numeric.iloc[val_idx].values, dtype=torch.float32).view(-1,1)

        model = TorchNet(X_tr.shape[1])

        optimizer = optim.AdamW(model.parameters(), lr=lr)
        loss_fn = nn.BCEWithLogitsLoss()

        model.train()

        for epoch in range(epochs):

            perm = torch.randperm(len(X_tr))

            for i in range(0, len(X_tr), batch_size):

                idx = perm[i:i+batch_size]

                xb = X_tr[idx]
                yb = y_tr[idx]

                preds = model(xb)

                loss = loss_fn(preds, yb)

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        model.eval()

        with torch.no_grad():

            preds = model(X_val)
            preds = torch.sigmoid(preds).numpy().flatten()

        auc = roc_auc_score(y_val.numpy(), preds)

        auc_scores.append(auc)

        trial.report(auc, step=len(auc_scores))
        if trial.should_prune():
            raise optuna.TrialPruned()

    return np.mean(auc_scores)


study = optuna.create_study(direction="maximize")

study.optimize(objective, n_trials=50)

print("Best params:", study.best_params)
print("Best CV AUC:", study.best_value)


best = study.best_params

X_t = torch.tensor(X_train.values, dtype=torch.float32)
y_t = torch.tensor(y_train_numeric.values, dtype=torch.float32).view(-1,1)

model = TorchNet(X_t.shape[1])

optimizer = optim.AdamW(model.parameters(), lr=best["lr"])
loss_fn = nn.BCEWithLogitsLoss()

for epoch in range(best["epochs"]):

    perm = torch.randperm(len(X_t))

    for i in range(0, len(X_t), best["batch_size"]):

        idx = perm[i:i+best["batch_size"]]

        xb = X_t[idx]
        yb = y_t[idx]

        preds = model(xb)

        loss = loss_fn(preds, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

model.eval()

X_test_t = torch.tensor(X_test.values, dtype=torch.float32)

with torch.no_grad():
    preds = torch.sigmoid(model(X_test_t)).numpy().flatten()

auc = roc_auc_score(y_test_numeric, preds)

print("Test AUC:", auc)

[I 2026-03-12 04:54:08,151] A new study created in memory with name: no-name-451b56cf-e849-454d-87f4-246cf52d7def
[I 2026-03-12 04:55:28,063] Trial 0 finished with value: 0.945543737110383 and parameters: {'lr': 0.0004733133746815864, 'batch_size': 512, 'epochs': 25}. Best is trial 0 with value: 0.945543737110383.
[I 2026-03-12 04:56:08,916] Trial 1 finished with value: 0.9438652510834886 and parameters: {'lr': 0.0004601376864747116, 'batch_size': 512, 'epochs': 14}. Best is trial 0 with value: 0.945543737110383.
[I 2026-03-12 04:57:09,310] Trial 2 finished with value: 0.9272807058440602 and parameters: {'lr': 0.00016746215160348853, 'batch_size': 512, 'epochs': 20}. Best is trial 0 with value: 0.945543737110383.
[I 2026-03-12 05:00:25,266] Trial 3 finished with value: 0.9548130290939664 and parameters: {'lr': 0.001867488046554872, 'batch_size': 128, 'epochs': 18}. Best is trial 3 with value: 0.9548130290939664.
[I 2026-03-12 05:01:40,825] Trial 4 finished with value: 0.948571202739109

KeyboardInterrupt: 